### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="kickstarter",
    dataset_year="2025",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://webrobots.io/kickstarter-datasets/",
    download_description="""
There exists data from Kaggle (https://www.kaggle.com/datasets/yashkantharia/kickstarter-campaign).
We use data from the original source (https://webrobots.io/kickstarter-datasets/) and downloaded the newest version at the time of writing.

wget https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2026-01-12T09_37_51_016Z.zip
mkdir -p local-data-warehouse/kickstarter && mv Kickstarter_2026-01-12T09_37_51_016Z.zip local-data-warehouse/kickstarter && mkdir -p local-data-warehouse/kickstarter/data_files && unzip local-data-warehouse/kickstarter/Kickstarter_2026-01-12T09_37_51_016Z.zip -d local-data-warehouse/kickstarter/data_files
""",
    # References
    academic_reference_bibtex=r"""@misc{webrobots2026kickstarter,
  title        = {Kickstarter Datasets},
  author       = {{Web Robots}},
  howpublished = {\url{https://webrobots.io/kickstarter-datasets/}},
  note         = {Accessed: 2026-01-25},
  year         = {2026},
  organization = {Web Robots}
}
""",
    academic_reference_bibtex_key="webrobots2026kickstarter",
    license="None",
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
Similar to the Kaggle task, we aim to predict whether a Kickstarter project will be funded successfully.

- The data we used only had 2 samples for 2026, so we stick to data until 2025.
- The original data comes in .csv files per scrape data. We first combine all files into one file.
- We remove all columns that are known only after the outcome and thus could leak information.
- The fx_rate seems to show the conversion rate at the time of crawling the data and not at the time of the project. Thus, if it is used as on Kaggle, the numbers are wrong. We use the "usd_exchange_rate" to transform the goal into USD currency.
- We drop the two entries with "disable_communication" as they point to other issues.
- We drop photo and video based references as we do not include these modalities.
- We decode the category and creator name  from JSON strings into usable columns.
- Note, the crawl does only include blurbs and not the full-text descriptions of the projects. Also some blurb repeat from similar projects or orders.
- We decode the profile blurb, when it exists.
- We decode the location display name from the location JSON string, which can include state and city name.
- The data contains spatial information (city, state, country). We do not decode this but leave it to the pipelines.
- We found 150 rows without location information. We drop these rows as it is unclear which issue caused this and how the rows' data might be affected by this.
- We drop duplicates (20%) which seems to have occurred from multiple scrapes of the same project.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="state",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="state",
    time_on="created_at", # could also be deadline to get more realistic data with projects for which we know the lab at "real" train time. But with our time splits, created_at is fine as well.
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
import json

data_dir = dataset_mold.path / "data_files"

# Read and concatenate all CSV files
df = pd.concat(
    (pd.read_csv(csv_file) for csv_file in data_dir.glob("*.csv")),
    ignore_index=True
)
print("Loaded data shape:", df.shape)

# Filter to successful or failed projects
df = df[df["state"].isin(["successful", "failed"])]

# Drop rows with missing location information
df = df[~df["location"].isna()]

# Convert goal to USD
non_usd_mask = df["currency"] != "USD"
df.loc[non_usd_mask, "goal"] = df.loc[non_usd_mask, "goal"] * df.loc[non_usd_mask, "usd_exchange_rate"]

# Use only full country name
df["country"] = df["country_displayable_name"]
df = df.drop(columns=["country_displayable_name"])
# drop disable_communication artifacts
df = df[df["disable_communication"] == False]
df = df.drop(columns=["disable_communication"])

# We do not include images for this benchmark, so we drop the related columns
df = df.drop(columns=["photo", "video"])

# Drop other columns
drop_columns = [
    # Remove target leakage columns
    "backers_count", "converted_pledged_amount", "is_in_post_campaign_pledging_phase",
    "percent_funded", "pledged", "static_usd_rate", "usd_pledged", "usd_type",
    "usd_exchange_rate", "state_changed_at",
    # Remove currency related columns not need anymore
    "currency", "currency_symbol", "currency_trailing_code",
    "current_currency", "fx_rate",
    # all unique
    "is_disliked", "is_launched", "is_liked", "is_starrable",
    # We got the date from created_at
    "id",
    # Just formating of name
    "slug",
    # No relation / not needed
    "source_url", "urls",
]
df = df.drop(columns=drop_columns)

# Decode category
df["main_category"] = df["category"].apply(lambda x: json.loads(x)["name"])
df["sub_category"] = df["category"].apply(lambda x: json.loads(x)["parent_name"] if "parent_name" in json.loads(x) else np.nan)
df = df.drop(columns=["category"])

# Decode profile blurb
def decode_profile(x):
    try:
        data = json.loads(x)
    except json.decoder.JSONDecodeError:
        raw_data = x.split(",")
        raw_data = [e for e in raw_data if e.startswith('"blurb":')]
        assert len(raw_data) == 1
        blurb_entry = raw_data[0]
        blurb_entry = blurb_entry.replace('"blurb":"', "").rstrip('"')
        if "null" in blurb_entry:
            return np.nan
        if blurb_entry == "":
            return np.nan
        return blurb_entry
    res = data["blurb"]
    if res == "":
        return np.nan
    return res

df["profile_blurb"] = df["profile"].apply(decode_profile)
df = df.drop(columns=["profile"])

# Decode creator name
def decode_name(x):
    try:
        data = json.loads(x)
    except json.decoder.JSONDecodeError:
        raw_data = x.split(",")
        raw_data = [e for e in raw_data if e.startswith('"name":')]
        assert len(raw_data) == 1
        name_entry = raw_data[0]
        name_entry = name_entry.replace('"name":"', "").rstrip('"')
        return name_entry
    return data["name"]
df["creator_name"] = df["creator"].apply(decode_name)
df = df.drop(columns=["creator"])

# Decode location
df["location_displayable_name"] = df["location"].apply(lambda x: json.loads(x)["displayable_name"])
df = df.drop(columns=["location"])

# Dtypes
as_string_cols = [
    "blurb",
    "name",
    "creator_name",
    "profile_blurb",
    "location_displayable_name",
]
as_date_cols = [
    "created_at",
    "launched_at",
    "deadline",
]
as_cat_type = [
    "prelaunch_activated",
    "spotlight",
    "staff_pick",
    "main_category",
    "sub_category",
    "state",
    "country",
]

for c in as_string_cols:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

for c in as_date_cols:
    df[c] = pd.to_datetime(df[c], unit="s")

df[as_cat_type] = df[as_cat_type].astype("category")

# Drop duplicates
df = df.drop_duplicates()

# Drop the 2 samples from 2026
df = df[df["created_at"].dt.year < 2026]

df = df.reset_index(drop=True)

Loaded data shape: (266279, 42)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 187,118
Columns: 16
Use sampling: False (sample size: 187,118)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['created_at', 'launched_at', 'name', 'blurb', 'deadline', 'creator_name', 'goal', 'profile_blurb', 'location_displayable_name', 'main_category']
Rows remaining as candidates after top-10 filter: 0 (of 187,118)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,blurb,country,created_at,deadline,goal,launched_at,name,prelaunch_activated,spotlight,staff_pick,state,main_category,sub_category,profile_blurb,creator_name,location_displayable_name
0,"A 1:1 remake of Valve Software's ""Meet the Medic"" animation. Featuring the original voice actors and professional set recreations.",Denmark,2025-03-11 20:56:49,2025-07-12 20:00:00,40911.74745,2025-06-12 19:18:55,Meet the REAL Medic: Team Fortress 2 Fan Film Remake,True,True,True,successful,Shorts,Film & Video,<NA>,ShorK,"Rennes, France"
1,"Tales from “Greenzone: Life in the Blocks”, “The Mighty KAAW The Crowmagnon” and special preview of ""Alice Lost In Wonderland""!",the United States,2024-08-31 16:38:25,2024-10-05 02:00:00,1000.00000,2024-09-23 15:22:56,FishTales Volumes 1 & 2,True,True,False,successful,Anthologies,Comics,<NA>,Fish Lee,"Vilonia, AR"
2,"An annual broadsheet newspaper full of art, comics, criticism, interviews. Our fourth issue is called THREE'S COMPANY.",the United States,2024-06-20 12:54:28,2024-10-04 03:59:00,30000.00000,2024-09-18 16:11:04,LAAB: The Newspaper of the Radical Imagination,True,True,True,successful,Anthologies,Comics,<NA>,Beehive Books,"Philadelphia, PA"
3,Looking for support for this calendar that will feature the top 12 cool classic cars based on on-line votes from the general public.,the United States,2014-07-02 23:33:12,2014-08-08 20:51:09,5500.00000,2014-07-09 20:51:09,The Gallatin County Cool Classic Car Calendar 2015,False,False,False,failed,Calendars,Publishing,<NA>,Dan Jessen,"Bozeman, MT"
4,The 2015 Dime Calendar will be the jump start to an amazing collection from All American Dimes.,the United States,2014-05-22 21:56:55,2014-07-31 07:38:35,5000.00000,2014-06-01 07:38:35,2015 Dime Calendar Project,False,False,False,failed,Calendars,Publishing,<NA>,All American Dimes,"Oakland, CA"


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,sub_category,category,4238.0,2.26,15.0,"Film & Video, Music, Publishing, Technology, Art, Food, Games, Fashion, Design, Crafts"
1,country,category,0.0,0.00,25.0,"the United States, the United Kingdom, Canada, Australia, Germany, Mexico, France, Italy, Spain, Hong Kong"
2,prelaunch_activated,category,0.0,0.00,2.0,"False, True"
3,spotlight,category,0.0,0.00,2.0,"True, False"
4,staff_pick,category,0.0,0.00,2.0,"False, True"
5,state,category,0.0,0.00,2.0,"successful, failed"
6,main_category,category,0.0,0.00,161.0,"Tabletop Games, Web, Product Design, Anthologies, Comedy, Documentary, Apparel, Comic Books, Jazz, Food Trucks"
7,created_at,datetime64[ns],0.0,0.00,187058.0,"2024-09-16 15:26:10, 2015-01-20 23:43:30, 2015-03-24 18:45:08, 2024-11-01 13:29:25, 2024-04-06 00:38:54, 2014-07-10 13:02:32, 2019-02-07 20:27:59, 2018-04-27 14:45:46, 2023-03-02 20:42:51, 2015-04-01 19:12:56"
8,deadline,datetime64[ns],0.0,0.00,177545.0,"2025-11-01 03:59:00, 2024-11-01 03:59:00, 2024-09-01 03:59:00, 2025-11-01 06:59:00, 2015-01-01 04:59:00, 2021-11-01 03:59:00, 2016-04-01 03:59:00, 2014-05-30 21:00:00, 2024-05-01 03:59:00, 2025-10-01 03:59:00"
9,launched_at,datetime64[ns],0.0,0.00,186897.0,"2025-07-01 14:00:01, 2024-09-17 15:00:08, 2025-10-07 14:00:01, 2024-03-07 16:36:02, 2023-11-01 17:00:05, 2024-03-18 11:00:13, 2024-11-21 15:00:20, 2017-05-16 16:46:13, 2025-10-28 13:59:33, 2025-09-30 12:30:10"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
goal,187118.0,34317.314037,1.005790e+06,0.01,135734937.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column                    rank                                                                        
blurb                     1     3D Printable STL Files, Terrain for Tabletop, Miniature, Role Pl...   
                          2     High-quality STL Files, 3D Pin-Up Printable Figures for Miniatur...   
                          3     A high-quality Figurines, STL file, 3D printable Presupported Minis   
                          4            Pre-Supported High Quality 3D Printable Figures (STL Files).   
                          5     3D Printable Miniatures for RPG & Tabletop Wargames (Pre-support...   
country                   1                                                       the United States   
                          2                                                      the United Kingdom   
                          3                                                                  Canada   
                          4                                                               Australia   
                          5                                                                 Germany   
created_at                1                                                     2024-09-16 15:26:10   
                          2                                                     2015-01-20 23:43:30   
                          3                                                     2015-03-24 18:45:08   
                          4                                                     2024-11-01 13:29:25   
                          5                                                     2024-04-06 00:38:54   
creator_name              1                                                    Microcosm Publishing   
                          2                                                                    Juan   
                          3                                                            Mike Hoffman   
                          4                                                                   David   
                          5                                                                  Daniel   
deadline                  1                                                     2025-11-01 03:59:00   
                          2                                                     2024-11-01 03:59:00   
                          3                                                     2024-09-01 03:59:00   
                          4                                                     2025-11-01 06:59:00   
                          5                                                     2015-01-01 04:59:00   
launched_at               1                                                     2025-07-01 14:00:01   
                          2                                                     2024-09-17 15:00:08   
                          3                                                     2025-10-07 14:00:01   
                          4                                                     2024-03-07 16:36:02   
                          5                                                     2023-11-01 17:00:05   
location_displayable_name 1                                                         Los Angeles, CA   
                          2                                                              London, UK   
                          3                                                            New York, NY   
                          4                                                             Chicago, IL   
                          5                                                            Brooklyn, NY   
main_category             1                                                          Tabletop Games   
                          2                                                                     Web   
                          3                                                  

In [8]:
# Target Distribution
target_df

,count,pct
state,,
successful,117148,62.61
failed,69970,37.39


## Task Curation

In [9]:
# Filter to year 2025

# Create a year-month column for grouping
df["year"] = df[task_mold.time_on].dt.to_period("Y")
monthly_totals = (
    df
    .groupby("year")
    .size()
    .rename("total_samples")
)
monthly_class_counts = (
    df
    .groupby(["year", task_mold.target_column_name])
    .size()
    .unstack(fill_value=0)
)
result = monthly_class_counts.join(monthly_totals)
result = result.sort_index()
result.index = result.index.to_timestamp()
result

/tmp/ipykernel_146137/543726850.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["year", task_mold.target_column_name])


,failed,successful,total_samples
year,,,
2009-01-01,5,118,123
2010-01-01,83,784,867
2011-01-01,290,2402,2692
2012-01-01,645,4644,5289
2013-01-01,950,5058,6008
2014-01-01,7590,8578,16168
2015-01-01,9856,8750,18606
2016-01-01,6941,7344,14285
2017-01-01,6374,7822,14196


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata

# Sort by time
df = df.sort_values(by=task_mold.time_on).reset_index(drop=True)

splits = {}

test_years = [
    "2023",
    "2024",
    "2025",
]
for i, month in enumerate(test_years):
    ref_date  = pd.Timestamp(month)
    train_index = df[
        df[task_mold.time_on] < ref_date
    ].index
    test_index = df[
        (df[task_mold.time_on].dt.year == ref_date.year)
    ].index
    splits[i] = {
        0: (train_index.tolist(), test_index.tolist())
    }

for s in splits:
    train_index, test_index = splits[s][0]
    print(f"Split {s}: Train size: {len(train_index)}, Test size: {len(test_index)}")
    assert df[task_mold.time_on].iloc[train_index].max() < df[task_mold.time_on].iloc[test_index].min()

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="""We try to create splits that simulate a model deployed to solve the task.

The official data is updated monthly but has not enough data per month to create large enough test splits. We opt for simulating a model that is refit every year to obtain a robust test set.

 We could simulate a model that is refitted every month, but this would need many splits. Moreover, data from just one month is not enough to create a robust test set. We instead simulate a model that is refit every year. This introduces the unrealistic downside of data shift across a month that would not exist in a real-world model. We create 3 test splits by 2023, 2024, and 2025 as test year. For each test split, we use all data before the test month as training data.
""",
    splits=splits,
    time_horizon=1,
    time_horizon_unit="years",
)

Split 0: Train size: 131836, Test size: 13056
Split 1: Train size: 144892, Test size: 21400
Split 2: Train size: 166292, Test size: 20826


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to kickstarter/019d30a1-68be-7d5c-b949-bbf4cef2bfe2
019d30a1-68be-7d5c-b949-bbf4cef2bfe2
66c5ae5181c6bc40a8f2d76b2531dc69c6b8dd9fc361e92beef832664913d4d1
